In [13]:
import os
import glob
import uproot
import mplhep as hep
import numpy as np
import awkward as ak
import boost_histogram as bh
import matplotlib.pyplot as plt
import json

In [14]:
def get_lumi_dict(grl_dir):
    grl_csvs = glob.glob(os.path.join(grl_dir, "*.csv"))
    lumi_dict = {}
    for grl in grl_csvs:
        with open(grl, 'r') as f:
            for i, line in enumerate(f):
                if i == 0: continue

                spline = line.split(",")
                lumi_rec = float(spline[3])
                run_num = int(spline[0])
                lumi_dict[run_num] = lumi_rec

    return lumi_dict

In [15]:
lumi_dict = get_lumi_dict("/cvmfs/faser.cern.ch/repo/sw/runlist/v8/")
runs = np.array(list(lumi_dict.keys()))
runs_2022 = runs[runs < 1e4]
runs_2023 = runs[runs > 1e4]
runs_2023 = runs_2023[runs_2023 < 1.2e4]
runs_2024 = runs[runs > 1.2e4]

In [16]:
low_lumi_runs = []
for run, lumi in lumi_dict.items():
    if lumi < 10: 
        low_lumi_runs.append(run)
print(low_lumi_runs)

[10417, 10419, 10443, 10540, 10572, 10600, 10602, 10747, 10799, 11214, 11461, 11463, 11478, 11480, 11486, 11491, 11706, 7733, 7734, 7802, 7833, 7835, 7836, 7848, 7849, 7971, 7984, 7987, 7988, 7989, 7990, 7998, 8034, 8036, 8037, 8038, 8039, 8040, 8041, 8090, 8096, 8097, 8098, 8099, 8104, 8105, 8106, 8108, 8109, 8110, 8111, 8112, 8113, 8114, 8243, 8247, 8253, 8295, 8651, 8652, 8655, 8658, 8713, 8714, 8734, 8751, 8901, 8904, 8910, 8932, 9048, 9053, 9056, 9066, 9171, 14587, 14588, 14589, 14590, 14597, 14733, 14766, 14776, 15075, 15146, 15181, 15257, 15259, 15260, 15267, 15421, 15455, 15553, 15676, 15678, 15679, 15868, 15975, 15993, 16522, 16648, 16669, 16716]


A list of bad runs can be found on the twiki here: https://twiki.cern.ch/twiki/bin/view/FASER/BadFASERRuns

In [17]:
bad_runs = [{'runs': [15075, 15146, 15181, 15257, 15259, 15260], "reason": "ATLAS lumi unrealiable"},
            {'runs': [10921], "reason": "ATLAS issues (salvagable)"},
            {'runs': [10526], "reason": "ATLAS lumi issue?"},
            {'runs': [15694], "reason": "Low-mu run, lumi unreliable"},
            {'runs': [11705], "reason": "TCL6 collimator not inserted"},
            {'runs': [16669], "reason": "100% deadtime"},
            {'runs': low_lumi_runs, "reason": "Run has < 10 /pb"}
           ]
    

In [18]:
processed_runs = []
files_2022 = glob.glob("output_2022/*.root")
files_2023 = glob.glob("output_2023/*.root")
files_2024 = glob.glob("output_2024/*.root")

all_files = files_2022 + files_2023 + files_2024

processed_runs += [int(os.path.basename(f).replace(".root", "")) for f in files_2022]
processed_runs += [int(os.path.basename(f).replace(".root", "")) for f in files_2023]
processed_runs += [int(os.path.basename(f).replace(".root", "")) for f in files_2024]

# print(processed_runs)

# print(list(set(processed_runs) - set(lumi_dict.keys())))

n_missing = 0
for r in processed_runs:
    if r not in lumi_dict.keys():
        print(f"Run {r} not in lumi dict")
        n_missing += 1
print(f"Number of missing runs = {n_missing}")

n_missing = 0
missing_lumi = 0
for r in lumi_dict.keys():
    if r not in processed_runs:
        missing_lumi += lumi_dict[r]
        print(f"Run {r} not in root output (lumi = {lumi_dict[r]:.2f} /pb)")
        if lumi_dict[r] >= 10:
            n_missing += 1
print(f"Number of missing runs = {n_missing} (lumi = {missing_lumi/1000:.2f} /fb)")

Number of missing runs = 0
Run 11214 not in root output (lumi = 0.00 /pb)
Run 7733 not in root output (lumi = 0.12 /pb)
Run 7734 not in root output (lumi = 0.29 /pb)
Run 7802 not in root output (lumi = 1.25 /pb)
Run 7833 not in root output (lumi = 7.18 /pb)
Run 7835 not in root output (lumi = 8.94 /pb)
Run 7836 not in root output (lumi = 0.00 /pb)
Run 7848 not in root output (lumi = 4.02 /pb)
Run 7849 not in root output (lumi = 2.25 /pb)
Run 7918 not in root output (lumi = 20.51 /pb)
Run 7930 not in root output (lumi = 44.44 /pb)
Run 7961 not in root output (lumi = 17.03 /pb)
Run 7971 not in root output (lumi = 2.04 /pb)
Run 7984 not in root output (lumi = 6.36 /pb)
Run 7985 not in root output (lumi = 32.41 /pb)
Run 7987 not in root output (lumi = 0.62 /pb)
Run 7988 not in root output (lumi = 0.66 /pb)
Run 7989 not in root output (lumi = 0.01 /pb)
Run 7990 not in root output (lumi = 0.64 /pb)
Run 7998 not in root output (lumi = 4.89 /pb)
Run 8006 not in root output (lumi = 70.70 /pb)
R

In [19]:
total_lumi_root = 0
lumi_2022 = 0
lumi_2023 = 0
lumi_2024 = 0
lumi_calonu = 0

n_runs_2022 = 0
n_runs_2023 = 0
n_runs_2024 = 0
n_runs_calonu = 0

for fpath in all_files:
    run = int(os.path.basename(fpath).replace(".root", ""))

    is_bad = False
    for entry in bad_runs:
        if run in entry['runs']: is_bad = True
    if is_bad: 
        # print(f"Skipping bad run: {run}")
        continue

    data = uproot.open(f"{fpath}:dq")
    lumi_arr = data['lumi'].arrays()
    lumi = lumi_arr['lumi'][0][0]

    if run in runs_2022: 
        lumi_2022 += lumi
        n_runs_2022 += 1
    if run in runs_2023: 
        lumi_2023 += lumi
        n_runs_2023 += 1
    if run in runs_2024: 
        lumi_2024 += lumi
        n_runs_2024 += 1

    if 15821 <= run	<= 16924:
        lumi_calonu += lumi
        n_runs_calonu += 1
    
    total_lumi_root += lumi
print(f"Lumi 2022: {lumi_2022:.2f} /fb ({n_runs_2022} runs)")
print(f"Lumi 2023: {lumi_2023:.2f} /fb ({n_runs_2023} runs)")
print(f"Lumi 2024: {lumi_2024:.2f} /fb ({n_runs_2024} runs)")
print(f"Lumi CaloNu: {lumi_calonu:.2f} /fb ({n_runs_calonu} runs)")
print(f"Total lumi in root files {total_lumi_root:.2f} /fb")


Lumi 2022: 27.21 /fb (68 runs)
Lumi 2023: 30.12 /fb (80 runs)
Lumi 2024: 121.95 /fb (247 runs)
Lumi CaloNu: 70.93 /fb (123 runs)
Total lumi in root files 179.28 /fb


In [20]:
total_lumi = sum(lumi_dict.values())
print(f"Total luminosity = {total_lumi/1000:.2f} /fb")

print("Following runs are marked for removal")
removed_lumi = 0
for entry in bad_runs:
    l = sum([lumi_dict[r] for r in entry['runs']])
    removed_lumi += l
    print(f"\t - {entry['reason']} - {l:.2f} /pb  {entry['runs']} ")
print(f"Total removed lumi: {removed_lumi/1000:.2f} /fb")
print(f"Total lumi after GRL: {(total_lumi - removed_lumi)/1000:.2f} /fb")

Total luminosity = 190.15 /fb
Following runs are marked for removal
	 - ATLAS lumi unrealiable - 1.63 /pb  [15075, 15146, 15181, 15257, 15259, 15260] 
	 - ATLAS issues (salvagable) - 120.81 /pb  [10921] 
	 - ATLAS lumi issue? - 46.16 /pb  [10526] 
	 - Low-mu run, lumi unreliable - 19.61 /pb  [15694] 
	 - TCL6 collimator not inserted - 772.49 /pb  [11705] 
	 - 100% deadtime - 0.05 /pb  [16669] 
	 - Run has < 10 /pb - 281.39 /pb  [10417, 10419, 10443, 10540, 10572, 10600, 10602, 10747, 10799, 11214, 11461, 11463, 11478, 11480, 11486, 11491, 11706, 7733, 7734, 7802, 7833, 7835, 7836, 7848, 7849, 7971, 7984, 7987, 7988, 7989, 7990, 7998, 8034, 8036, 8037, 8038, 8039, 8040, 8041, 8090, 8096, 8097, 8098, 8099, 8104, 8105, 8106, 8108, 8109, 8110, 8111, 8112, 8113, 8114, 8243, 8247, 8253, 8295, 8651, 8652, 8655, 8658, 8713, 8714, 8734, 8751, 8901, 8904, 8910, 8932, 9048, 9053, 9056, 9066, 9171, 14587, 14588, 14589, 14590, 14597, 14733, 14766, 14776, 15075, 15146, 15181, 15257, 15259, 15260, 15

In [21]:
def print_latex_lumi_table(data_file_list, bad_runs, num_cols=4):

    table_str = r"\begin{tabular}{"
    for _ in range(num_cols):
        table_str += r"cS@{\hskip 0.5in}"
    table_str += "}\n"
    table_str += r"\toprule"
    table_str += "\n"
    for i in range(num_cols):
        table_str += r"{Run} & {Lumi. [\si{\per\pico\barn}]}"
        if i > 0: table_str += " & "
    table_str += r"\\"
    table_str += "\n"
    table_str  += r"\midrule"

#     table_str = \
# r""""
# \begin{tabular}{cS@{\hskip 0.5in}cS@{\hskip 0.5in}cS@{\hskip 0.5in}cS}
# \toprule
# {Run} & {Lumi. [\si{\per\pico\barn}]} & {Run} & {Lumi. [\si{\per\pico\barn}]} & {Run} & {Lumi. [\si{\per\pico\barn}]} & {Run} & {Lumi. [\si{\per\pico\barn}]} \\
# \midrule
# """
    i = 0
    body_completed = False
    for fpath in data_file_list:
        run = int(os.path.basename(fpath).replace(".root", ""))

        is_bad = False
        for entry in bad_runs:
            if run in entry['runs']: is_bad = True
        if is_bad:
            continue
        i += 1

        data = uproot.open(f"{fpath}:dq")
        lumi_arr = data['lumi'].arrays()
        lumi = lumi_arr['lumi'][0][0]
        
        table_str += f"{run} & {lumi*1000:.2f}"

        if i % num_cols != 0: 
            table_str += " & "
            body_completed= False
        else: 
            table_str += r" \\" + "\n"
            body_completed = True


    while True:
        if body_completed: break
        if i % num_cols != 0: 
            table_str += " & "
            i += 1
        elif not body_completed: 
            table_str += r" \\" + "\n"
            break

    table_str += r"\bottomrule"
    table_str += "\n"
    table_str += r"\end{tabular}"
    print(table_str)

In [22]:
print_latex_lumi_table(files_2022, bad_runs, num_cols=4)

\begin{tabular}{cS@{\hskip 0.5in}cS@{\hskip 0.5in}cS@{\hskip 0.5in}cS@{\hskip 0.5in}}
\toprule
{Run} & {Lumi. [\si{\per\pico\barn}]}{Run} & {Lumi. [\si{\per\pico\barn}]} & {Run} & {Lumi. [\si{\per\pico\barn}]} & {Run} & {Lumi. [\si{\per\pico\barn}]} & \\
\midrule8752 & 10.29 & 8725 & 150.89 & 8717 & 52.17 & 8736 & 415.71 \\
8730 & 237.25 & 8733 & 486.14 & 8749 & 108.94 & 8775 & 514.33 \\
8777 & 465.23 & 8843 & 23.93 & 8846 & 38.21 & 8799 & 280.10 \\
8841 & 112.48 & 8837 & 128.74 & 8729 & 391.05 & 8847 & 203.05 \\
8728 & 429.72 & 8731 & 353.34 & 8798 & 582.10 & 8850 & 580.20 \\
8933 & 76.47 & 8926 & 32.12 & 8834 & 641.55 & 8912 & 29.19 \\
8947 & 208.07 & 8928 & 39.84 & 8944 & 273.58 & 8915 & 399.40 \\
8848 & 725.36 & 8952 & 121.19 & 8955 & 386.62 & 8948 & 363.64 \\
8930 & 632.81 & 8906 & 563.51 & 8970 & 585.54 & 9058 & 445.43 \\
8969 & 182.03 & 8981 & 558.42 & 8977 & 164.10 & 8975 & 254.16 \\
8913 & 665.40 & 8943 & 799.58 & 8949 & 751.47 & 8916 & 644.07 \\
8985 & 358.17 & 8950 & 409.84 

In [23]:
print_latex_lumi_table(files_2023, bad_runs)

\begin{tabular}{cS@{\hskip 0.5in}cS@{\hskip 0.5in}cS@{\hskip 0.5in}cS@{\hskip 0.5in}}
\toprule
{Run} & {Lumi. [\si{\per\pico\barn}]}{Run} & {Lumi. [\si{\per\pico\barn}]} & {Run} & {Lumi. [\si{\per\pico\barn}]} & {Run} & {Lumi. [\si{\per\pico\barn}]} & \\
\midrule10422 & 10.77 & 10604 & 177.63 & 10555 & 66.12 & 10538 & 74.83 \\
10601 & 41.83 & 10424 & 13.53 & 10426 & 13.92 & 10693 & 58.12 \\
10694 & 85.48 & 10733 & 330.68 & 10695 & 129.46 & 10698 & 116.51 \\
10605 & 251.95 & 10847 & 307.25 & 11041 & 99.08 & 10881 & 290.27 \\
10829 & 547.45 & 10796 & 400.81 & 10701 & 344.25 & 10615 & 201.62 \\
10888 & 52.68 & 10699 & 304.00 & 10729 & 327.56 & 10884 & 92.19 \\
10732 & 348.24 & 10792 & 647.98 & 10700 & 312.23 & 10738 & 319.96 \\
10794 & 684.25 & 10859 & 159.05 & 10775 & 684.71 & 11081 & 416.49 \\
11082 & 67.91 & 10857 & 211.89 & 11079 & 244.28 & 11038 & 58.29 \\
10743 & 473.94 & 11088 & 189.62 & 10928 & 737.43 & 11093 & 540.57 \\
11068 & 236.59 & 10956 & 728.74 & 11471 & 15.81 & 11070 & 41

In [24]:
print_latex_lumi_table(files_2024, bad_runs, num_cols=4)

\begin{tabular}{cS@{\hskip 0.5in}cS@{\hskip 0.5in}cS@{\hskip 0.5in}cS@{\hskip 0.5in}}
\toprule
{Run} & {Lumi. [\si{\per\pico\barn}]}{Run} & {Lumi. [\si{\per\pico\barn}]} & {Run} & {Lumi. [\si{\per\pico\barn}]} & {Run} & {Lumi. [\si{\per\pico\barn}]} & \\
\midrule14765 & 39.00 & 14767 & 54.76 & 14593 & 13.19 & 14645 & 19.24 \\
14618 & 11.43 & 14647 & 52.90 & 14743 & 276.64 & 14777 & 186.51 \\
14644 & 17.42 & 14989 & 183.99 & 14810 & 64.46 & 14905 & 32.23 \\
14904 & 39.77 & 14763 & 50.71 & 15023 & 178.56 & 14764 & 110.10 \\
14906 & 199.02 & 14985 & 235.07 & 14760 & 225.39 & 14771 & 174.52 \\
14976 & 218.53 & 15289 & 36.03 & 15015 & 599.44 & 15016 & 680.90 \\
15424 & 79.65 & 14954 & 239.58 & 14971 & 238.33 & 15396 & 56.49 \\
15055 & 723.99 & 14973 & 285.50 & 15287 & 218.92 & 14769 & 323.51 \\
15269 & 491.74 & 14758 & 28.74 & 14977 & 480.32 & 15034 & 375.71 \\
14974 & 354.93 & 15391 & 533.74 & 15024 & 724.68 & 15387 & 609.12 \\
14981 & 683.28 & 15050 & 520.48 & 15398 & 163.76 & 14975 & 666

In [25]:
for i, bad_run in enumerate(bad_runs):
    lost_lumi = sum([lumi_dict[r] for r in bad_run['runs']])
    bad_runs[i]['lumi'] = lost_lumi
print(bad_runs)
with open('bad_runs.json', 'w') as f:
    json.dump(bad_runs, f)

[{'runs': [15075, 15146, 15181, 15257, 15259, 15260], 'reason': 'ATLAS lumi unrealiable', 'lumi': 1.629}, {'runs': [10921], 'reason': 'ATLAS issues (salvagable)', 'lumi': 120.81}, {'runs': [10526], 'reason': 'ATLAS lumi issue?', 'lumi': 46.156}, {'runs': [15694], 'reason': 'Low-mu run, lumi unreliable', 'lumi': 19.609}, {'runs': [11705], 'reason': 'TCL6 collimator not inserted', 'lumi': 772.485}, {'runs': [16669], 'reason': '100% deadtime', 'lumi': 0.049}, {'runs': [10417, 10419, 10443, 10540, 10572, 10600, 10602, 10747, 10799, 11214, 11461, 11463, 11478, 11480, 11486, 11491, 11706, 7733, 7734, 7802, 7833, 7835, 7836, 7848, 7849, 7971, 7984, 7987, 7988, 7989, 7990, 7998, 8034, 8036, 8037, 8038, 8039, 8040, 8041, 8090, 8096, 8097, 8098, 8099, 8104, 8105, 8106, 8108, 8109, 8110, 8111, 8112, 8113, 8114, 8243, 8247, 8253, 8295, 8651, 8652, 8655, 8658, 8713, 8714, 8734, 8751, 8901, 8904, 8910, 8932, 9048, 9053, 9056, 9066, 9171, 14587, 14588, 14589, 14590, 14597, 14733, 14766, 14776, 15075,

A list of the the different run periods for 2024 can be found here: https://docs.google.com/spreadsheets/d/1nnYFcmhVieSHI5XAVhPiW1K6CoGYGxv2YPchwL0sqH4/edit?gid=0#gid=0

In [26]:
run_splits = []
header = []
with open("2024RunSplits.csv", 'r') as f:
    for i, line in enumerate(f):
        if i == 0: 
            header = line.split(',')
            continue
        
        split_dict = {}
        spline = line.split(',')
        for key, value in zip(header, spline):
            split_dict[key.strip()] = value.strip()   
        run_splits.append(split_dict)

with open("2024RunSplits.json", 'w') as f:
    json.dump(run_splits, f)